In [ ]:
!pip install -q sentencepiece langchain-text-splitters huggingface-hub

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
import torch, gc, time, textwrap
from transformers import MBartForConditionalGeneration, MBartTokenizer, T5ForConditionalGeneration, T5Tokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter

def clear_memory():
  '''очистка GPU памяти между моделями'''
  gc.collect()
  if torch.cuda.is_available():
    torch.cuda.empty_cache()


def clean_summary(text):
  '''убирает дублирующиеся предложения'''
  sentences=text.split('. ')
  seen=set()
  unique=[]
  for s in sentences:
    if s not in seen and len(s)>5:
      seen.add(s)
      unique.append(s)
  return '. '.join(unique)

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
def summarize(text, model, tokenizer, model_type='mbart', max_length=150, min_length=50):
  '''суммаризация чанка'''

  if not text or len(text)<20:
    return text

  inputs=tokenizer(text, max_length=1024, truncation=True, padding=True, return_tensors='pt').to(device)

  gen_params={
      'max_length': max_length,
      'min_length': min_length,
      'num_beams':5,
      'early_stopping':True,
      'no_repeat_ngram_size':3,
      'repetition_penalty':1.4,
      'temperature':0.7,
      'do_sample':False
  }

  #для mbart указываем язык
  if model_type=='mbart':
    gen_params['forced_bos_token_id']=tokenizer.lang_code_to_id['ru_RU']

  with torch.no_grad():
    summary_ids=model.generate(inputs['input_ids'], **gen_params)

  summary=tokenizer.decode(
      summary_ids[0],
      skip_special_tokens=True,
      clean_up_tokenization_spaces=True
  )

  summary=clean_summary(summary)
  return summary

In [ ]:
def hierarchical_summarize(text, model, tokenizer, model_type='mbart', chunk_size=800, chunk_overlap=100):
  '''разбивает текст на чанки, суммаризирует каждый, затем объединяет'''

  if len(text)<200:
    return summarize(text, model, tokenizer, model_type, max_length=100, min_length=30)

  #разбиваем на чанки
  text_splitter=RecursiveCharacterTextSplitter(
      chunk_size=chunk_size,
      chunk_overlap=chunk_overlap,
      length_function=len,
      separators=['\n','\n\n', '. ', '!', '?', ';', ' ','']
  )

  chunks=text_splitter.split_text(text)
  print(f'текст разбит на {len(chunks)} частей')

  # суммаризация каждого чанка
  chunk_summaries=[]
  for i, chunk in enumerate(chunks):
    if len(chunk)>100:
      summary=summarize(chunk, model, tokenizer, model_type, max_length=80, min_length=20)
      if summary and len(summary)>10:
        chunk_summaries.append(summary)
      else:
        first_sentence=chunk.split('.')[0]+'.'
        chunk_summaries.append(first_sentence)
    else:
      chunk_summaries.append(chunk)

    time.sleep(0.05)

  combined=' '.join(chunk_summaries)
  if len(combined)>400:
    final=summarize(combined, model, tokenizer, model_type, max_length=150, min_length=50)
    return final

  return combined

In [ ]:
test_texts = {
    "Новость (политика)": """
Президент России Владимир Путин провел совещание с членами правительства, посвященное развитию экономики в условиях санкционного давления. Глава государства отметил, что, несмотря на внешние ограничения, российская экономика продолжает расти. По словам президента, важно обеспечить технологический суверенитет страны и поддержать отечественных производителей. Министр экономического развития доложил о росте ВВП на 3.5% по итогам прошлого года. Также обсуждались вопросы импортозамещения в промышленности и сельском хозяйстве. Путин поручил правительству подготовить дополнительные меры поддержки бизнеса и продолжить работу над улучшением инвестиционного климата в регионах.
""",

    "Юридический текст": """
В соответствии со статьей 450 Гражданского кодекса Российской Федерации, изменение и расторжение договора возможны по соглашению сторон, если иное не предусмотрено настоящим Кодексом, другими законами или договором. По требованию одной из сторон договор может быть изменен или расторгнут по решению суда только при существенном нарушении договора другой стороной, а также в иных случаях, предусмотренных настоящим Кодексом, другими законами или договором. Существенным признается нарушение договора одной из сторон, которое влечет для другой стороны такой ущерб, что она в значительной степени лишается того, на что была вправе рассчитывать при заключении договора.
""",

    "Научно-популярный текст": """
Нейронные сети представляют собой один из ключевых методов искусственного интеллекта, который основан на обучении на примерах. Этот подход можно сравнить с тем, как ребенок учится различать кошек и собак, просматривая множество фотографий. Нейросеть функционирует аналогичным образом: ей демонстрируются тысячи примеров, и она самостоятельно выявляет закономерности в данных. Глубокое обучение представляет собой особую разновидность нейронных сетей, характеризующуюся наличием множества слоев. На первом уровне происходит распознавание простейших элементов - линий и контуров, на втором уровне формируются более сложные образы, а на верхних этажах распознаются целостные объекты.
""",

    "Большая новость: ИИ в России": """
Президент России Владимир Путин подписал указ о новой национальной стратегии развития искусственного интеллекта до 2030 года. Документ, опубликованный на официальном портале правовой информации, предполагает увеличение финансирования отрасли в 5 раз по сравнению с текущим уровнем. Согласно стратегии, к 2030 году не менее 80 процентов государственных служащих должны использовать системы ИИ в своей работе, а объем рынка отечественного ПО в сфере ИИ должен достичь 600 миллиардов рублей. Эксперты отмечают, что документ был разработан при участии Сбербанка, Яндекса и Минцифры, и он станет основой для нового национального проекта "Экономика данных".

Глава Минцифры Максут Шадаев на совещании у президента сообщил, что в рамках новой стратегии будет создано не менее 15 центров компетенций в ведущих российских вузах. Первые такие центры уже открылись на базе МГУ, ИТМО и Сколтеха, где начнут подготовку специалистов по большим генеративным моделям. Шадаев подчеркнул, что особенно остро стоит вопрос с кадрами: текущая потребность отрасли оценивается в 30 тысяч специалистов высокого уровня, и закрыть этот дефицит планируется за счет целевого обучения и переподготовки сотрудников госкомпаний.

Министр также анонсировал создание единой платформы для сбора и обработки больших данных, необходимых для обучения нейросетей. Платформа объединит обезличенные данные из государственных информационных систем, медицинских учреждений и промышленных предприятий. Доступ к ней получат российские разработчики на безвозмездной основе, что, по мнению экспертов, должно ускорить создание конкурентоспособных AI-решений. Разработкой платформы занимается госкорпорация "Росатом", имеющая необходимые компетенции в области хранения и обработки больших объемов информации.

Между тем, крупнейшие российские технологические компании уже представили свои новые разработки в сфере ИИ. Сбер анонсировал третье поколение своей нейросетевой модели GigaChat, которая, по заявлению компании, превосходит предыдущую версию по качеству генерации текстов на русском языке в два раза. В свою очередь, Яндекс сообщил о внедрении генеративных нейросетей во все свои сервисы, включая поиск, карты и такси. Компания также открыла доступ к своей флагманской модели YandexGPT 5 для разработчиков, предоставив льготные условия для стартапов. Эксперты рынка сходятся во мнении, что конкуренция между двумя технологическими гигантами стимулирует развитие всей отрасли.

Однако участники рынка обращают внимание и на существующие проблемы. По словам генерального директора АНО "Цифровая экономика" Сергея Плуготаренко, ключевым вызовом остается дефицит вычислительных мощностей. Для обучения больших моделей требуются тысячи графических процессоров, закупка которых осложнена санкционными ограничениями. В связи с этим правительство рассматривает возможность субсидирования отечественных производителей серверного оборудования и создания распределенной сети дата-центров с использованием российских процессоров. Первый такой дата-центр планируется запустить в Казани до конца следующего года, его мощность составит 100 петафлопс.

Параллельно с развитием технологий ведется работа над нормативным регулированием. В Совете Федерации создана рабочая группа по этике искусственного интеллекта, которая разрабатывает кодекс ответственного использования нейросетей. Сенаторы отмечают, что необходимо найти баланс между стимулированием инноваций и защитой граждан от потенциальных угроз, связанных с распространением дипфейков и автоматизированным принятием решений. Первый зампред комитета по конституционному законодательству Ирина Рукавишникова заявила, что законопроект о регулировании генеративных нейросетей может быть внесен в Госдуму уже в весеннюю сессию. Документ, в частности, предусматривает обязательную маркировку контента, созданного искусственным интеллектом.
""",

    "Спортивные новости": """
В городе на Неве состоялось открытие шестого по счету командного турнира "Кубок Первого канала". Соревнования высокого уровня проходят с 21 по 22 марта 2026 года. В борьбе за престижный трофей сошлись две сборные — команда "Москва" и команда "Санкт-Петербург". Одним из главных событий первого дня стало выступление чемпионки России Аделии Петросян, которая продемонстрировала высочайший уровень мастерства.
Восемнадцатилетняя фигуристка, представляющая тренерскую школу Этери Тутберидзе и сотрудничающая с Даниилом Глейхенгаузом, чисто исполнила четверной тулуп в короткой программе. Это выступление состоялось 21 марта 2026 года. Данный элемент считается одним из самых сложных в женском катании, и на всех остальных официальных соревнованиях исполнения четверных прыжков женщинами в короткой программе не допускается. Однако регламент текущего сезона "Кубка Первого канала" сделал исключение и разрешил спортсменкам включать квады в программу, чем и воспользовалась Петросян.
Аделия Петросян является трехкратной чемпионкой России и трехкратной победительницей Финала Гран-при страны. Ранее в этом году она представляла свою страну на 25-й зимней Олимпиаде 2026 года в Италии, где заняла шестое место. На турнире в Санкт-Петербурге она выступает в качестве капитана красной команды "Москва". Для Петросян это не первый опыт руководства коллективом. Год назад, в 2025 году, она уже исполняла обязанности капитана на прыжковом чемпионате России, который также принимал Петербург. Тогда москвичи соперничали с питерской командой. После своего проката спортсменка поделилась с журналистами впечатлениями от турнира. Она отметила невероятную атмосферу соревнований и удовольствие от выступлений на данной арене. Также фигуристка призналась, что несколько раз пыталась отказаться от роли капитана, однако команда оказала ей поддержку, что помогло принять окончательное решение.
"""
}

In [ ]:
def test_model(model_name, model, tokenizer, model_type):
  '''тестирование модели на всех текстах'''

  print(f'\nтестирование модели: {model_name}\n')
  for text_name, text_content in test_texts.items():
    print(f'текст: {text_name}')
    print(f'длина текста: {len(text_content)} символов')
    print(f'оригинал (начало):')
    print(text_content[:150]+'...')

    try:
      start_time=time.time()

      res=hierarchical_summarize(
          text_content,
          model,
          tokenizer,
          model_type=model_type,
          chunk_size=800,
          chunk_overlap=100
      )

      t=time.time()-start_time

      print(f'\nсуммаризация (сжатый текст):\n')
      print(res)
      print(f'\nсжатие: {len(text_content)} -> {len(res)} ({len(text_content)/len(res):.1f}x)')
      print(f'\nвремя: {t:.2f} сек')

    except Exception as e:
      print(f'ошибка: {e}')

    print('-'*100)

In [ ]:
clear_memory()

base_model_name_mbart = "facebook/mbart-large-cc25"
finetuned_model_name_mbart = "IlyaGusev/mbart_ru_sum_gazeta"
print(f'\nмодель 1: {finetuned_model_name_mbart}')

model_mbart=MBartForConditionalGeneration.from_pretrained(base_model_name_mbart).to(device)

#загружаем дообученные веса Ильи Гусева
state_dict = torch.hub.load_state_dict_from_url(
    "https://huggingface.co/IlyaGusev/mbart_ru_sum_gazeta/resolve/main/pytorch_model.bin",
    map_location=device
)
model_mbart.load_state_dict(state_dict, strict=False)

tokenizer_mbart = MBartTokenizer.from_pretrained(base_model_name_mbart)

tokenizer_mbart.src_lang='ru_RU'
tokenizer_mbart.tgt_lang='ru_RU'

test_model('MBart (IlyaGusev)', model_mbart, tokenizer_mbart, 'mbart')

In [ ]:
clear_memory()

model_t5_name='IlyaGusev/rut5_base_sum_gazeta'

print(f'\nмодель 2: {model_t5_name}')

model_t5=T5ForConditionalGeneration.from_pretrained(model_t5_name).to(device)
tokenizer_t5=T5Tokenizer.from_pretrained(model_t5_name)

test_model('T5 (IlyaGusev)', model_t5, tokenizer_t5, 't5')

In [ ]:
# интерактивный ввод текста

while True:
  text=input('текст: ')
  if text.lower()=='выйти':
    break

  if len(text)<10:
    print('слишком короткий текст для суммаризации (сжатия)')
    continue

  start_time=time.time()
  try:
    res=hierarchical_summarize(text, model_mbart, tokenizer_mbart, 'mbart')
    print(f'\nсуммаризация:\n{res}\n')
    print(f'сжатие: {len(text)} -> {len(res)} ({len(text)/len(res):.1f}x)')
    print(f'время: {time.time()-start_time:.1f} сек\n')
  except Exception as e:
    print(f'ошибка: {e}\n')

In [ ]:
# телеграм бот

!pip install -q python-telegram-bot nest-asyncio

import nest_asyncio
nest_asyncio.apply()
from google.colab import userdata
from telegram import Update
from telegram.ext import Application, CommandHandler, MessageHandler, filters, ContextTypes

TOKEN = userdata.get('BOT_TOKEN')

async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text("отправь текст, я сделаю суммаризацию")

async def handle_text(update: Update, context: ContextTypes.DEFAULT_TYPE):
    text = update.message.text
    print(f"получен текст: {text[:50]}...")

    msg = await update.message.reply_text("обрабатываю...")

    try:
        result = hierarchical_summarize(text, model_mbart, tokenizer_mbart, 'mbart')
        print(f"отправлен результат: {len(result)} символов\n")
        await msg.edit_text(f"-> {result}")
    except Exception as e:
        print(f"ошибка: {e}\n")
        await msg.edit_text(f"Ошибка: {e}")

app = Application.builder().token(TOKEN).build()
app.add_handler(CommandHandler("start", start))
app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_text))

print("бот запущен!\n")
app.run_polling()